# Descriptive analyses by irrigation scheme and soil depth

Reproduces Figure 4.11 and Figure 4.12 of the thesis using MIR-predicted EC$_{1:2}$.

The notebook contains only the analysis retained in the final thesis. Exploratory alternatives, superseded cells, personal paths, saved outputs, and unpublished figure-formatting trials have been removed.

In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent

DATA_DIR = PROJECT_DIR / "data"
OUTPUTS_DIR = PROJECT_DIR / "outputs"

SOIL_FILE = DATA_DIR / "soil_data.xlsx"
SOIL_SHEET = "PredictionNewSamplesSalinity_Fi"

if not SOIL_FILE.exists():
    raise FileNotFoundError(
        f"Input file not found: {SOIL_FILE}\n"
        "Place the soil dataset in the data folder and name it 'soil_data.xlsx', "
        "or update SOIL_FILE in this notebook."
    )

import warnings
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from IPython.display import display

warnings.filterwarnings("ignore", category=FutureWarning)

OUT_DIR = OUTPUTS_DIR / "01_descriptive_figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_excel(SOIL_FILE, sheet_name=SOIL_SHEET)
df.columns = df.columns.str.strip()

required_columns = [
    "SITE", "Layer", "EC 1:2 (dS/cm)", "ESP est",
    "TotalBacteria", "TotalFungi", "TYPES_BACTERIA", "TYPES_FUNGI",
    "Ksat", "TotalMacrofaunaBiomass (g)"
]
missing = [column for column in required_columns if column not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df["Zone"] = df["SITE"].astype(str).str.strip().str.upper()
df = df[df["Zone"].isin(["PERKERRA", "TANA"])].copy()

numeric_columns = [
    "Layer", "EC 1:2 (dS/cm)", "ESP est",
    "TotalBacteria", "TotalFungi", "TYPES_BACTERIA", "TYPES_FUNGI",
    "Ksat", "TotalMacrofaunaBiomass (g)"
]
for column in numeric_columns:
    df[column] = pd.to_numeric(df[column], errors="coerce")

df = df.rename(columns={
    "EC 1:2 (dS/cm)": "EC12",
    "ESP est": "ESP",
    "TYPES_BACTERIA": "BacterialRichness",
    "TYPES_FUNGI": "FungalRichness",
    "Ksat": "Kfs",
    "TotalMacrofaunaBiomass (g)": "MacrofaunaBiomass",
})

depth_map = {1: "0–20 cm", 2: "20–50 cm", 3: "50–80 cm"}
depth_order = ["0–20 cm", "20–50 cm", "50–80 cm"]
zone_order = ["PERKERRA", "TANA"]
palette = {"PERKERRA": "#4C72B0", "TANA": "#DD8452"}

df["Depth"] = df["Layer"].map(depth_map)

print("Dataset shape:", df.shape)
print(pd.crosstab(df["Zone"], df["Depth"]))


In [ ]:
analysis_variables = [
    "EC12", "ESP", "TotalBacteria", "TotalFungi",
    "BacterialRichness", "FungalRichness"
]

summary = (
    df.groupby(["Zone", "Depth"], observed=True)[analysis_variables]
      .agg(["count", "median", "mean", "std", "min", "max"])
      .round(3)
)
summary.to_excel(OUT_DIR / "descriptive_statistics_by_scheme_and_depth.xlsx")
display(summary)


In [ ]:
# Figure 4.11: depth-related variation in salinity, sodicity and microbial indicators
sns.set_style("whitegrid")
sns.set_context("notebook", font_scale=1.45)

font_axis = 24
font_tick = 20
font_legend = 21
font_panel = 27

plot_variables = [
    "EC12", "ESP", "TotalBacteria", "TotalFungi",
    "BacterialRichness", "FungalRichness"
]
labels = {
    "EC12": r"EC$_{1:2}$ (dS m$^{-1}$)",
    "ESP": "ESP (%)",
    "TotalBacteria": "Total bacterial abundance",
    "TotalFungi": "Total fungal abundance",
    "BacterialRichness": "Bacterial richness",
    "FungalRichness": "Fungal richness",
}

fig, axes = plt.subplots(3, 2, figsize=(19.5, 21.5))
axes = axes.ravel()
panel_labels = ["a)", "b)", "c)", "d)", "e)", "f)"]

for index, variable in enumerate(plot_variables):
    ax = axes[index]

    sns.boxplot(
        data=df, x="Depth", y=variable, hue="Zone",
        order=depth_order, hue_order=zone_order,
        palette=palette, width=0.7, fliersize=0, ax=ax
    )
    sns.stripplot(
        data=df, x="Depth", y=variable, hue="Zone",
        order=depth_order, hue_order=zone_order,
        dodge=True, alpha=0.4, size=3.5, linewidth=0,
        palette=palette, ax=ax
    )

    if ax.get_legend() is not None:
        ax.get_legend().remove()

    ax.set_ylabel(labels[variable], fontsize=font_axis)
    ax.set_xlabel("Soil depth" if index in [4, 5] else "", fontsize=font_axis)
    ax.tick_params(axis="both", labelsize=font_tick)

fig.subplots_adjust(
    left=0.08, right=0.98, bottom=0.07, top=0.91,
    wspace=0.22, hspace=0.25
)

handles, legend_labels = axes[0].get_legend_handles_labels()
fig.legend(
    handles[:2], legend_labels[:2], title="Zone",
    loc="upper left", bbox_to_anchor=(0.04, 1.035),
    frameon=False, fontsize=font_legend, title_fontsize=font_legend
)

for index, ax in enumerate(axes):
    position = ax.get_position()
    fig.text(
        position.x0 - 0.010, position.y1 + 0.010, panel_labels[index],
        fontsize=font_panel, fontweight="bold", ha="left", va="bottom"
    )

for extension in ["png", "pdf"]:
    figure_path = OUT_DIR / f"figure_4_11_depth_scheme_boxplots.{extension}"
    plt.savefig(
        figure_path,
        dpi=600 if extension == "png" else None,
        bbox_inches="tight"
    )

plt.show()


In [ ]:
# Figure 4.12: surface-layer hydraulic conductivity and macrofauna biomass
surface = df[df["Layer"] == 1].copy()

fig, axes = plt.subplots(1, 2, figsize=(15.5, 6.8))
surface_specs = [
    ("Kfs", r"$K_{fs}$ (cm min$^{-1}$)", "a)"),
    ("MacrofaunaBiomass", "Macrofauna biomass (g)", "b)"),
]

for ax, (variable, ylabel, panel) in zip(axes, surface_specs):
    sns.boxplot(
        data=surface, x="Zone", y=variable,
        order=zone_order, palette=palette,
        width=0.6, fliersize=0, linewidth=1.2, ax=ax
    )
    sns.stripplot(
        data=surface, x="Zone", y=variable,
        order=zone_order, color="black",
        alpha=0.5, size=4, ax=ax
    )
    ax.set_xlabel("Zone", fontsize=18)
    ax.set_ylabel(ylabel, fontsize=18)
    ax.tick_params(axis="both", labelsize=15)

fig.subplots_adjust(
    left=0.07, right=0.98, bottom=0.12, top=0.92, wspace=0.20
)

for ax, (_, _, panel) in zip(axes, surface_specs):
    position = ax.get_position()
    fig.text(
        position.x0 - 0.005, position.y1 + 0.008, panel,
        fontsize=19, fontweight="bold", ha="left", va="bottom"
    )

surface_summary = (
    surface.groupby("Zone", observed=True)[["Kfs", "MacrofaunaBiomass"]]
           .agg(["count", "median", "mean", "std", "min", "max"])
           .round(4)
)
surface_summary.to_excel(OUT_DIR / "surface_layer_descriptive_statistics.xlsx")

for extension in ["png", "pdf"]:
    figure_path = OUT_DIR / f"figure_4_12_surface_variables.{extension}"
    plt.savefig(
        figure_path,
        dpi=600 if extension == "png" else None,
        bbox_inches="tight"
    )

plt.show()
display(surface_summary)
